In [0]:
# ============================================
# Cell 1: 创建初始 Delta 表
# ============================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

merge_path = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimProduct_Practice/"

# 初始数据：3 个产品
initial_data = [
    (1, "Product A", 100.00, "2024-01-01", "9999-12-31", 1),
    (2, "Product B", 200.00, "2024-01-01", "9999-12-31", 1),
    (3, "Product C", 300.00, "2024-01-01", "9999-12-31", 1),
]

dim_product = (spark.createDataFrame(
    initial_data,
    ["ProductKey", "ProductName", "ListPrice", "valid_from", "valid_to", "is_current"]
).withColumn("valid_from", F.to_date("valid_from"))
 .withColumn("valid_to", F.to_date("valid_to"))
 .withColumn("is_current", F.col("is_current").cast("int"))   # ⭐ 确保是 int
)

dim_product.write.format("delta").mode("overwrite").save(merge_path)

print("✅ Initial table created (3 rows)")
spark.read.format("delta").load(merge_path).show()

✅ Initial table created (3 rows)
+----------+-----------+---------+----------+----------+----------+
|ProductKey|ProductName|ListPrice|valid_from|  valid_to|is_current|
+----------+-----------+---------+----------+----------+----------+
|         1|  Product A|    100.0|2024-01-01|9999-12-31|         1|
|         2|  Product B|    200.0|2024-01-01|9999-12-31|         1|
|         3|  Product C|    300.0|2024-01-01|9999-12-31|         1|
+----------+-----------+---------+----------+----------+----------+



In [0]:
# ============================================
# Cell 2: 模拟源数据更新
# ============================================

incoming_data = [
    (1, "Product A", 120.00),   # 变更（100 → 120）
    (2, "Product B", 200.00),   # 未变
    (4, "Product D", 400.00),   # 新增
]

incoming = (spark.createDataFrame(
    incoming_data,
    ["ProductID", "ProductName", "ListPrice"]
).withColumn("ListPrice", F.col("ListPrice").cast("decimal(19,4)"))
)

incoming.show()

+---------+-----------+---------+
|ProductID|ProductName|ListPrice|
+---------+-----------+---------+
|        1|  Product A| 120.0000|
|        2|  Product B| 200.0000|
|        4|  Product D| 400.0000|
+---------+-----------+---------+



In [0]:
# ============================================
# Cell 3: Step 1 - 关闭变更的旧记录
# ============================================

target = DeltaTable.forPath(spark, merge_path)

(target.alias("t")
    .merge(
        incoming.alias("s"),
        "t.ProductKey = s.ProductID AND t.is_current = 1"
    )
    .whenMatchedUpdate(
        condition="t.ListPrice <> s.ListPrice",
        set={
            "valid_to": "current_date()",
            "is_current": "0"
        }
    )
    .execute()
)

print("✅ Step 1: Old records closed")
spark.read.format("delta").load(merge_path).orderBy("ProductKey").show()

✅ Step 1: Old records closed
+----------+-----------+---------+----------+----------+----------+
|ProductKey|ProductName|ListPrice|valid_from|  valid_to|is_current|
+----------+-----------+---------+----------+----------+----------+
|         1|  Product A|    100.0|2024-01-01|2026-09-27|         0|
|         2|  Product B|    200.0|2024-01-01|9999-12-31|         1|
|         3|  Product C|    300.0|2024-01-01|9999-12-31|         1|
+----------+-----------+---------+----------+----------+----------+



In [0]:
# ============================================
# Cell 4: Step 2 - 插入新版本
# ============================================

# 找出目标表里当前有效的 ProductKey
target_current = (spark.read.format("delta").load(merge_path)
    .filter(F.col("is_current") == 1)
    .select(F.col("ProductKey").alias("CurrentProductID"))
)

# 找出 incoming 里，目标表没有当前版本的行 → 需要插入新版本
to_insert = (incoming.alias("s")
    .join(target_current.alias("t"), 
          F.col("s.ProductID") == F.col("t.CurrentProductID"), 
          how="left_anti")
    .withColumn("ProductKey", F.col("ProductID") + 100)
    .withColumn("ListPrice", F.col("ListPrice").cast("double"))    # ⭐ 改成 double
    .withColumn("valid_from", F.current_date())
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(1).cast("int"))
    .select("ProductKey", "ProductName", "ListPrice", "valid_from", "valid_to", "is_current")
)

to_insert.show()
to_insert.write.format("delta").mode("append").save(merge_path)

print("✅ Step 2: New versions inserted")

+----------+-----------+---------+----------+----------+----------+
|ProductKey|ProductName|ListPrice|valid_from|  valid_to|is_current|
+----------+-----------+---------+----------+----------+----------+
|       101|  Product A|    120.0|2026-09-27|9999-12-31|         1|
|       104|  Product D|    400.0|2026-09-27|9999-12-31|         1|
+----------+-----------+---------+----------+----------+----------+

✅ Step 2: New versions inserted


In [0]:
# ============================================
# Cell 5: 验证 SCD Type 2 结果
# ============================================

final_df = spark.read.format("delta").load(merge_path).orderBy("ProductKey")
final_df.show()

print(f"Total rows: {final_df.count()}")
# 期望：5 行

+----------+-----------+---------+----------+----------+----------+
|ProductKey|ProductName|ListPrice|valid_from|  valid_to|is_current|
+----------+-----------+---------+----------+----------+----------+
|         1|  Product A|    100.0|2024-01-01|2026-09-27|         0|
|         2|  Product B|    200.0|2024-01-01|9999-12-31|         1|
|         3|  Product C|    300.0|2024-01-01|9999-12-31|         1|
|       101|  Product A|    120.0|2026-09-27|9999-12-31|         1|
|       104|  Product D|    400.0|2026-09-27|9999-12-31|         1|
+----------+-----------+---------+----------+----------+----------+

Total rows: 5


In [0]:
# 查看历史版本
display(spark.sql(f"DESCRIBE HISTORY delta.`{merge_path}`"))

# 读取 MERGE 之前的版本
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(merge_path)
print("Version 0 (before MERGE):")
df_v0.show()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-09-27T08:20:07.000Z,140877466452824,yonggang.chen@lincolnuni.ac.nz,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3935446796285536),7f2d92fe-2a4b-4e1f-b3d5-39383286748a,0927-054326-5fb7c09z-v2n,2,WriteSerializable,false,"Map(numFiles -> 2, numOutputRows -> 2, numOutputBytes -> 3764)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-27T08:20:06.000Z,140877466452824,yonggang.chen@lincolnuni.ac.nz,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3935446796285536),7e6b358e-7b65-49b8-b241-9a110625b62d,0927-054326-5fb7c09z-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3868, p25FileSize -> 1971, numDeletionVectorsRemoved -> 1, minFileSize -> 1971, numAddedFiles -> 1, maxFileSize -> 1971, p75FileSize -> 1971, p50FileSize -> 1971, numAddedBytes -> 1971)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-27T08:20:05.000Z,140877466452824,yonggang.chen@lincolnuni.ac.nz,MERGE,"Map(predicate -> [""((ProductKey#20446L = ProductID#20434L) AND (is_current#20451 = 1))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (ListPrice#20448 = cast(ListPrice#20438 as double))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(3935446796285536),7e6b358e-7b65-49b8-b241-9a110625b62d,0927-054326-5fb7c09z-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1900, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 2341, materializeSourceTimeMs -> 93, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1054, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1172)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-27T08:20:01.000Z,140877466452824,yonggang.chen@lincolnuni.ac.nz,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3935446796285536),297bf10b-dcda-4022-beae-73481a906222,0927-054326-5fb7c09z-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1968)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


Version 0 (before MERGE):
+----------+-----------+---------+----------+----------+----------+
|ProductKey|ProductName|ListPrice|valid_from|  valid_to|is_current|
+----------+-----------+---------+----------+----------+----------+
|         1|  Product A|    100.0|2024-01-01|9999-12-31|         1|
|         2|  Product B|    200.0|2024-01-01|9999-12-31|         1|
|         3|  Product C|    300.0|2024-01-01|9999-12-31|         1|
+----------+-----------+---------+----------+----------+----------+

